In [41]:
# -*- coding: utf-8 -*-
"""
etl_utils.py (Phiên bản thông minh - auto detect header)
--------------------------------------------------------
Tự động phát hiện dòng tiêu đề trong báo cáo công việc Excel.
Hỗ trợ header 1 hoặc 2 dòng (ô gộp "Thời gian thực hiện / Từ / Đến").

Chức năng chính:
- Detect header tự động.
- Detect sections (A.I, A.II, A.III, B.I, B.II).
- Chuẩn hóa ngày (ISO) + nhận diện tần suất ("Hàng ngày", "Hàng tuần"...).
- Mapping STT 3.1 → subtask (parent=3).
- Trả JSON chuẩn MongoDB schema.
"""

from __future__ import annotations
import json, re
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import pandas as pd


# ========= Hằng số & mapping =========
SECTION_AI, SECTION_AII, SECTION_AIII, SECTION_BI, SECTION_BII = "A.I", "A.II", "A.III", "B.I", "B.II"

FREQ_MAP = {
    "hàng ngày": "hang_ngay",
    "hang ngay": "hang_ngay",
    "hàng tuần": "hang_tuan",
    "hang tuan": "hang_tuan",
    "hàng tháng": "hang_thang",
    "hang thang": "hang_thang",
}

TASK_TYPE_MAP = {
    "kế hoạch": "ke_hoach",
    "ke hoach": "ke_hoach",
    "đột xuất": "dot_xuat",
    "dot xuat": "dot_xuat",
    "thường xuyên": "thuong_xuyen",
    "thuong xuyen": "thuong_xuyen",
    "hàng ngày": "thuong_xuyen",
    "hang ngay": "thuong_xuyen",
}

COLUMN_PATTERNS = {
    "stt":       [r"^stt\b"],
    "title":     [r"\bcông việc\b", r"\bcong viec\b", r"\bnội dung\b", r"\bnoi dung\b"],
    "task_type": [r"\bloại công việc\b", r"\bloai cong viec\b", r"\bphân loại\b", r"\bphan loai\b"],
    "time_from": [r"\bthời gian thực hiện.*từ\b", r"\bthoi gian thuc hien.*tu\b", r"\btừ\b", r"\btu\b"],
    "time_to":   [r"\bthời gian thực hiện.*đến\b", r"\bthoi gian thuc hien.*den\b", r"\bđến\b", r"\bden\b"],
    "result":    [r"\bkết quả thực hiện\b", r"\bket qua thuc hien\b"],
    "difficulty":[r"\bđánh giá khó khăn\b", r"\bdanh gia kho khan\b", r"tồn đọng|ton dong|thuận lợi|thuan loi"],
    "cost_vnd":  [r"\bchi phí.*(vnđ|vnd)\b", r"\bchi phi\b"]
}


# ========= Dataclasses =========
@dataclass
class EmployeeMeta:
    _id: str
    employee_code: str
    full_name: str
    department: Optional[str] = None
    position: Optional[str]   = None
    email: Optional[str]      = None

@dataclass
class ReportMeta:
    _id: str
    employee_id: str
    period_from: str
    period_to: str
    report_month_key: str
    created_at: str


# ========= Header Auto Detection =========
def detect_header_row(xlsx_path, sheet_name, keywords=("stt", "công việc", "loại", "thời gian"),max_check=30):
    """Tìm dòng chứa tiêu đề dựa theo từ khóa."""
    df_preview = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None, nrows=max_check)
    for i in range(len(df_preview)):
        row_values = " ".join(str(v).lower() for v in df_preview.iloc[i].dropna().astype(str))
        score = sum(1 for k in keywords if k in row_values)
        if score >= 2:  # ít nhất 2 từ khóa
            return i
    return 0


# ========= Column normalization =========
def _normalize_cols(df_mi: pd.DataFrame) -> pd.DataFrame:
    def flat(col_tuple):
        parts = [str(p) for p in col_tuple if 'Unnamed' not in str(p)]
        name = re.sub(r"\s+", " ", " ".join(parts)).strip()
        return name
    df = df_mi.copy()
    df.columns = [flat(c) if isinstance(c, tuple) else str(c) for c in df_mi.columns]
    return df

def _pick_column(cols: List[str], patterns: List[str]) -> Optional[str]:
    lowered = [(c, c.lower()) for c in cols]
    for pat in patterns:
        rx = re.compile(pat, flags=re.IGNORECASE)
        for original, low in lowered:
            if rx.search(low):
                return original
    return None

def detect_columns(df: pd.DataFrame) -> Dict[str, Optional[str]]:
    cols = list(df.columns)
    return {
        "stt":       _pick_column(cols, COLUMN_PATTERNS["stt"]),
        "title":     _pick_column(cols, COLUMN_PATTERNS["title"]),
        "task_type": _pick_column(cols, COLUMN_PATTERNS["task_type"]),
        "from":      _pick_column(cols, COLUMN_PATTERNS["time_from"]),
        "to":        _pick_column(cols, COLUMN_PATTERNS["time_to"]),
        "result":    _pick_column(cols, COLUMN_PATTERNS["result"]),
        "difficulty":_pick_column(cols, COLUMN_PATTERNS["difficulty"]),
        "cost":      _pick_column(cols, COLUMN_PATTERNS["cost_vnd"]),
    }


# ========= Parsers =========
def parse_stt(value) -> Optional[float]:
    try:
        return float(str(value).strip())
    except Exception:
        return None

def map_task_type(value) -> Optional[str]:
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    for k, v in TASK_TYPE_MAP.items():
        if k in s:
            return v
    return "ke_hoach"

def detect_frequency(value) -> Optional[str]:
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    for k, v in FREQ_MAP.items():
        if k in s:
            return v
    return None

def to_iso_date_or_none(value) -> Optional[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)) or (isinstance(value, str) and value.strip() == ""):
        return None
    if isinstance(value, (pd.Timestamp, )):
        return pd.to_datetime(value).date().isoformat()
    s = str(value).strip()
    if s.lower() in {"nat", "nan", "none", ""}:
        return None
    if re.search(r"h(à|a)ng", s.lower()):
        return None
    for fmt in ("%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return pd.to_datetime(s, format=fmt).date().isoformat()
        except Exception:
            pass
    try:
        return pd.to_datetime(float(s), unit="D", origin="1899-12-30").date().isoformat()
    except Exception:
        return None

def parse_int_or_none(value) -> Optional[int]:
    if pd.isna(value):
        return None
    s = re.sub(r"[^\d]", "", str(value))
    return int(s) if s else None


# ========= Section Detection =========
def detect_sections(df: pd.DataFrame, col_stt: str):
    def idxs_of(label: str):
        return df.index[(df[col_stt].astype(str).str.strip() == label)].tolist()
    i_rows   = idxs_of("I")
    ii_rows  = idxs_of("II")
    iii_rows = idxs_of("III")
    b_rows   = df.index[df[col_stt].astype(str).str.startswith("B.")].tolist()

    sections = {}
    eval_rows = []

    if i_rows and ii_rows:
        for r in range(i_rows[0] + 1, ii_rows[0]):
            sections[r] = "A.I"
    if ii_rows and iii_rows:
        for r in range(ii_rows[0] + 1, iii_rows[0]):
            sections[r] = "A.II"
    if iii_rows:
        start_eval = iii_rows[0] + 1
        end_eval = b_rows[0] if b_rows else None
        if end_eval is not None and end_eval > start_eval:
            eval_rows = list(range(start_eval, end_eval))
    if len(i_rows) > 1 and len(ii_rows) > 1:
        for r in range(i_rows[1] + 1, ii_rows[1]):
            sections[r] = "B.I"
    if len(ii_rows) > 1:
        for r in range(ii_rows[1] + 1, len(df)):
            sections[r] = "B.II"
    return sections, eval_rows


# ========= Read sheet (Auto header) =========
def read_report_sheet(xlsx_path: str, sheet_name: str):
    """Đọc sheet và tự động phát hiện header (1 hoặc 2 dòng)."""
    header_rows = detect_header_row(xlsx_path, sheet_name)
    df_mi = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=[header_rows,header_rows+1])
    df = _normalize_cols(df_mi)
    colmap = detect_columns(df)
    missing = [k for k in ("stt", "title") if not colmap.get(k)]
    if missing:
        raise ValueError(f"Không tìm được cột bắt buộc: {missing}")
    return df, colmap


# ========= Extract entities =========
def extract_entities(df: pd.DataFrame, colmap, report_meta, employee_meta):
    sections, eval_rows = detect_sections(df, colmap["stt"])
    tasks, subtasks, evaluations = [], [], []
    for idx, row in df.iterrows():
        if idx in eval_rows or idx not in sections:
            continue
        sec = sections[idx]
        stt = parse_stt(row[colmap["stt"]])
        title = None if pd.isna(row[colmap["title"]]) else str(row[colmap["title"]]).strip()
        if stt is None or not title or title.lower() == "nan":
            continue
        task_type = map_task_type(row[colmap["task_type"]]) if colmap["task_type"] else "ke_hoach"
        freq      = detect_frequency(row[colmap["from"]])  if colmap["from"] else None
        time_from = to_iso_date_or_none(row[colmap["from"]]) if colmap["from"] else None
        time_to   = to_iso_date_or_none(row[colmap["to"]])   if colmap["to"]   else None
        result    = None if (not colmap["result"] or pd.isna(row[colmap["result"]])) else str(row[colmap["result"]]).strip()
        diff      = None if (not colmap["difficulty"] or pd.isna(row[colmap["difficulty"]])) else str(row[colmap["difficulty"]]).strip()
        cost      = parse_int_or_none(row[colmap["cost"]]) if colmap["cost"] else None

        base = {
            "report_id": report_meta._id,
            "section": sec,
            "stt": float(stt),
            "title": title,
            "task_type": task_type,
            "time_from": time_from,
            "time_to": time_to,
            "frequency": freq,
            "result": result,
            "difficulty_note": diff,
            "cost_vnd": cost,
            "project_id": None,
            "parent_task_id": None,
            "tags": []
        }
        if float(stt).is_integer():
            base["_id"] = f"T{sec.replace('.', '')}_{int(stt):03d}"
            tasks.append(base)
        else:
            parent_int = int(float(stt))
            base["_id"] = f"ST{sec.replace('.', '')}_{str(stt).replace('.', '_')}"
            base["parent_task_id"] = f"T{sec.replace('.', '')}_{parent_int:03d}"
            subtasks.append(base)

    for idx in eval_rows:
        row = df.iloc[idx]
        stt_raw = str(row[colmap["stt"]]).strip()
        if stt_raw not in {"1", "2"}:
            continue
        topic = "hang_ngay" if stt_raw == "1" else "chuyen_mon"
        content = None if pd.isna(row[colmap["title"]]) else str(row[colmap["title"]]).strip()
        if content:
            evaluations.append({
                "_id": f"EV_{stt_raw}",
                "report_id": report_meta._id,
                "topic": topic,
                "content": content
            })
    return {
        "employees": [employee_meta.__dict__],
        "reports":   [report_meta.__dict__],
        "projects":  [],
        "tasks":     tasks,
        "subtasks":  subtasks,
        "evaluations": evaluations
    }


In [42]:
xlsx_path = "input/202509_Bao cao cong viec_thinhdv.xlsx"
sheet_name = "Th8-T9"
df, colmap = read_report_sheet(xlsx_path, sheet_name)
colmap

{'stt': 'Stt',
 'title': 'Công việc',
 'task_type': 'Loại công việc',
 'from': 'Thời gian thực hiện Từ',
 'to': 'Thời gian thực hiện Đến',
 'result': 'Kết quả thực hiện',
 'difficulty': 'Đánh giá khó khăn thuận lợi/Tồn đọng',
 'cost': None}

In [27]:
# -*- coding: utf-8 -*-
"""
Trích xuất báo cáo công việc tháng từ Excel -> JSON chuẩn NoSQL (MongoDB)
Đã xử lý header 2 dòng (Thời gian thực hiện / Từ / Đến)
Tự nhận diện tần suất (Hàng ngày, Hàng tuần...) và chuyển đổi ngày ISO.
"""

import pandas as pd
import re, json
from datetime import datetime

# ========= 1. Đọc file Excel =========
xlsx_path = "input/202509_Bao cao cong viec_thinhdv.xlsx"
sheet_name = "Th8-T9"

# Đọc với header 2 dòng (dòng 6 & 7)
df_mi = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=[6,7])


# Gộp tên cột 2 dòng lại thành 1 chuỗi
def flatten_col(col_tuple):
    parts = [str(p) for p in col_tuple if  'Unnamed' not in str(p)]
    name = " ".join(parts)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

df = df_mi.copy()
df.columns = [flatten_col(c) for c in df_mi.columns]
print(df.columns)
# ========= 2. Xác định tên cột =========
def col_like(*keywords):
    for c in df.columns:
        low = c.lower()
        if all(k.lower() in low for k in keywords):
            return c
    return None

col_stt = col_like("stt")
col_title = col_like("công việc")
col_type = col_like("loại công việc")
col_from = col_like("thời gian thực hiện", "từ")
col_to = col_like("thời gian thực hiện", "đến")
col_result = col_like("kết quả thực hiện")
col_diff = col_like("đánh giá khó khăn")
col_cost = col_like("chi phí thực hiện")

# ========= 3. Xác định vùng section (A.I, A.II, A.III, B.I, B.II) =========
def row_idxs_of(label):
    return df.index[(df[col_stt].astype(str).str.strip() == label)].tolist()

i_rows = row_idxs_of('I')
ii_rows = row_idxs_of('II')
iii_rows = row_idxs_of('III')
b_rows = df.index[df[col_stt].astype(str).str.startswith('B.')].tolist()

sections = {}

# A.I
start_ai = i_rows[0] + 1
end_ai = ii_rows[0]
for idx in range(start_ai, end_ai):
    sections[idx] = "A.I"
print(sections)

# A.II
start_aii = ii_rows[0] + 1
end_aii = iii_rows[0]
for idx in range(start_aii, end_aii):
    sections[idx] = "A.II"

# A.III
start_eval = iii_rows[0] + 1
end_eval = b_rows[0] if b_rows else start_eval
eval_rows = list(range(start_eval, end_eval))

# B.I
start_bi = i_rows[1] + 1
end_bi = ii_rows[1]
for idx in range(start_bi, end_bi):
    sections[idx] = "B.I"

# B.II
start_bii = ii_rows[1] + 1
end_bii = len(df)
for idx in range(start_bii, end_bii):
    sections[idx] = "B.II"

# ========= 4. Hàm chuẩn hóa dữ liệu =========
def parse_stt_num(s):
    try:
        return float(str(s).strip())
    except:
        return None

def map_task_type(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "kế hoạch" in s or "ke hoach" in s:
        return "ke_hoach"
    if "đột xuất" in s or "dot xuat" in s:
        return "dot_xuat"
    if "thường xuyên" in s or "thuong xuyen" in s or "hàng ngày" in s or "hang ngay":
        return "thuong_xuyen"
    return "ke_hoach"

def detect_frequency(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "hàng ngày" in s or "hang ngay" in s:
        return "hang_ngay"
    if "hàng tuần" in s or "hang tuan" in s:
        return "hang_tuan"
    if "hàng tháng" in s or "hang thang" in s:
        return "hang_thang"
    return None

def to_date_iso_or_none(val):
    if pd.isna(val):
        return None
    if isinstance(val, (pd.Timestamp, )):
        return pd.to_datetime(val).date().isoformat()
    s = str(val).strip()
    if s.lower() in ("nat", "nan", "none", ""):
        return None
    if re.search(r'h(à|a)ng', s.lower()):
        return None
    for fmt in ("%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return pd.to_datetime(s, format=fmt).date().isoformat()
        except:
            pass
    try:
        return pd.to_datetime(float(s), unit='D', origin='1899-12-30').date().isoformat()
    except:
        return None

def parse_int(val):
    if pd.isna(val):
        return None
    s = re.sub(r'[^\d]', '', str(val))
    return int(s) if s else None

# ========= 5. Trích xuất Task / Subtask / Evaluation =========
tasks, subtasks, evaluations = [], [], []
df.head(5)
for idx, row in df.iterrows():
    if idx in eval_rows or idx not in sections:
        continue
    sec = sections[idx]
    stt = parse_stt_num(row[col_stt])
    title = None if pd.isna(row[col_title]) else str(row[col_title]).strip()
    print(title)
    break
    if not stt or not title or title.lower() == "nan":
        continue

    task_type = map_task_type(row[col_type])
    freq = detect_frequency(row[col_from])
    time_from = to_date_iso_or_none(row[col_from])
    time_to = to_date_iso_or_none(row[col_to])
    result = None if pd.isna(row[col_result]) else str(row[col_result]).strip()
    diff = None if pd.isna(row[col_diff]) else str(row[col_diff]).strip()
    cost = parse_int(row[col_cost]) if col_cost else None

    base = {
        "report_id": "REP202509_THINHDV",
        "section": sec,
        "stt": float(stt),
        "title": title,
        "task_type": task_type,
        "time_from": time_from,
        "time_to": time_to,
        "frequency": freq,
        "result": result,
        "difficulty_note": diff,
        "cost_vnd": cost,
        "project_id": None,
        "parent_task_id": None,
        "tags": []
    }

    if float(stt).is_integer():
        base["_id"] = f"T{sec.replace('.', '')}_{int(stt):03d}"
        tasks.append(base)
    else:
        parent_int = int(float(stt))
        base["_id"] = f"ST{sec.replace('.', '')}_{str(stt).replace('.', '_')}"
        base["parent_task_id"] = f"T{sec.replace('.', '')}_{parent_int:03d}"
        subtasks.append(base)

# Đánh giá (A.III)
for idx in eval_rows:
    row = df.iloc[idx]
    stt = str(row[col_stt]).strip()
    if stt not in ['1', '2']:
        continue
    topic = "hang_ngay" if stt == '1' else "chuyen_mon"
    content = None if pd.isna(row[col_title]) else str(row[col_title]).strip()
    if content:
        evaluations.append({
            "_id": f"EV_{stt}",
            "report_id": "REP202509_THINHDV",
            "topic": topic,
            "content": content
        })

# ========= 6. Tạo JSON cuối =========
employee = {
    "_id": "EMP001",
    "employee_code": "THINHDV",
    "full_name": "Đỗ Văn Thịnh",
    "department": "Phòng HC-CNTT",
    "position": "Nhân viên IT",
    "email": None
}

report = {
    "_id": "REP202509_THINHDV",
    "employee_id": "EMP001",
    "period_from": "2025-08-20",
    "period_to": "2025-09-20",
    "report_month_key": "202509",
    "created_at": datetime.utcnow().isoformat() + "Z"
}

output = {
    "employees": [employee],
    "reports": [report],
    "projects": [],
    "tasks": tasks,
    "subtasks": subtasks,
    "evaluations": evaluations
}

# ========= 7. Ghi file JSON =========

out_path = "output/report_extracted1.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ Đã tạo:", out_path)
print("Tasks:", len(tasks), " | Subtasks:", len(subtasks), " | Evaluations:", len(evaluations))


Index(['Stt', 'Công việc', 'Loại công việc', 'Thời gian thực hiện Từ',
       'Thời gian thực hiện Đến', 'Kết quả thực hiện',
       'Đánh giá khó khăn thuận lợi/Tồn đọng', 'Đ/G KQ'],
      dtype='object')
{1: 'A.I', 2: 'A.I', 3: 'A.I', 4: 'A.I', 5: 'A.I', 6: 'A.I', 7: 'A.I', 8: 'A.I', 9: 'A.I', 10: 'A.I', 11: 'A.I', 12: 'A.I', 13: 'A.I', 14: 'A.I', 15: 'A.I', 16: 'A.I', 17: 'A.I', 18: 'A.I', 19: 'A.I', 20: 'A.I', 21: 'A.I', 22: 'A.I', 23: 'A.I', 24: 'A.I', 25: 'A.I', 26: 'A.I', 27: 'A.I', 28: 'A.I', 29: 'A.I', 30: 'A.I', 31: 'A.I', 32: 'A.I', 33: 'A.I', 34: 'A.I', 35: 'A.I', 36: 'A.I', 37: 'A.I', 38: 'A.I', 39: 'A.I', 40: 'A.I', 41: 'A.I', 42: 'A.I', 43: 'A.I', 44: 'A.I', 45: 'A.I', 46: 'A.I', 47: 'A.I', 48: 'A.I', 49: 'A.I', 50: 'A.I', 51: 'A.I', 52: 'A.I', 53: 'A.I', 54: 'A.I', 55: 'A.I', 56: 'A.I', 57: 'A.I', 58: 'A.I', 59: 'A.I', 60: 'A.I'}
Vận hành hệ thống phần mềm
✅ Đã tạo: output/report_extracted1.json
Tasks: 0  | Subtasks: 0  | Evaluations: 2


In [33]:
def detect_sections(df, col_stt):
    """
    Xác định vùng dữ liệu thuộc section (A.I, A.II, A.III, B.I, B.II)
    từ cột 'Stt' trong file Excel báo cáo công việc.

    Trả về:
        sections (dict): {row_index: section_code}
        eval_rows (list): các dòng thuộc phần Đánh giá (A.III)
    """
    def row_idxs_of(label):
        return df.index[(df[col_stt].astype(str).str.strip() == label)].tolist()

    i_rows = row_idxs_of('I')
    ii_rows = row_idxs_of('II')
    iii_rows = row_idxs_of('III')
    b_rows = df.index[df[col_stt].astype(str).str.startswith('B.')].tolist()

    sections = {}

    # A.I: từ sau "I" đầu tiên đến trước "II" đầu tiên
    if i_rows and ii_rows:
        for idx in range(i_rows[0] + 1, ii_rows[0]):
            sections[idx] = "A.I"

    # A.II: từ sau "II" đầu tiên đến trước "III"
    if ii_rows and iii_rows:
        for idx in range(ii_rows[0] + 1, iii_rows[0]):
            sections[idx] = "A.II"

    # A.III: Đánh giá chung
    start_eval = iii_rows[0] + 1 if iii_rows else None
    end_eval = b_rows[0] if b_rows else None
    eval_rows = list(range(start_eval, end_eval)) if start_eval and end_eval else []

    # B.I: từ sau "I" thứ hai đến trước "II" thứ hai
    if len(i_rows) > 1 and len(ii_rows) > 1:
        for idx in range(i_rows[1] + 1, ii_rows[1]):
            sections[idx] = "B.I"

    # B.II: từ sau "II" thứ hai đến hết file
    if len(ii_rows) > 1:
        for idx in range(ii_rows[1] + 1, len(df)):
            sections[idx] = "B.II"

    return sections, eval_rows
sections, eval_rows = detect_sections(df, col_stt)
print(json.dumps(sections,indent=4))

{
    "1": "A.I",
    "2": "A.I",
    "3": "A.I",
    "4": "A.I",
    "5": "A.I",
    "6": "A.I",
    "7": "A.I",
    "8": "A.I",
    "9": "A.I",
    "10": "A.I",
    "11": "A.I",
    "12": "A.I",
    "13": "A.I",
    "14": "A.I",
    "15": "A.I",
    "16": "A.I",
    "17": "A.I",
    "18": "A.I",
    "19": "A.I",
    "20": "A.I",
    "21": "A.I",
    "22": "A.I",
    "23": "A.I",
    "24": "A.I",
    "25": "A.I",
    "26": "A.I",
    "27": "A.I",
    "28": "A.I",
    "29": "A.I",
    "30": "A.I",
    "31": "A.I",
    "32": "A.I",
    "33": "A.I",
    "34": "A.I",
    "35": "A.I",
    "36": "A.I",
    "37": "A.I",
    "38": "A.I",
    "39": "A.I",
    "40": "A.I",
    "41": "A.I",
    "42": "A.I",
    "43": "A.I",
    "44": "A.I",
    "45": "A.I",
    "46": "A.I",
    "47": "A.I",
    "48": "A.I",
    "49": "A.I",
    "50": "A.I",
    "51": "A.I",
    "52": "A.I",
    "53": "A.I",
    "54": "A.I",
    "55": "A.I",
    "56": "A.I",
    "57": "A.I",
    "58": "A.I",
    "59": "A.I",
    

In [ ]:
str.lower("Nhân viên") in str.lower("Nhân Viên IT")

True

: 